**Cell #01**

# RAG11 Nutrition — Stage 2: Ask Sample Questions

Runs a small set of hand-picked nutrition questions end to end through the
retrieval + generation pipeline built in Stage 1:

1. Embed the question with Voyage AI (`input_type="query"`, the counterpart
   to the `input_type="document"` used when the child chunks themselves were
   embedded in `stage1_2_eda_load_chunks.ipynb` -- Voyage's embeddings are
   asymmetric, tuned differently for the query side vs. the document side).
2. Retrieve the best-matching child chunks via the `match_rag11_child_chunks`
   RPC (`sql/create_sql_tables.sql`). At least 3 chunks always go to the LLM,
   so no answer is ever generated from a single, possibly-unlucky match.
3. Ask Claude to answer strictly from those retrieved excerpts. When a
   question has a clean Yes/No answer, the reply leads with a
   `Short answer: Yes` / `Short answer: No` line before the full
   explanation; open-ended questions just get the full explanation.
4. Every answer also reports the source page numbers behind it (parsed from
   each retrieved chunk's `[Source: ... | Pages X-Y]` contextual header), so
   an answer can always be checked against the original PDF.

**Before running this notebook**: `stage1_2_eda_load_chunks.ipynb` must
already have loaded and embedded your chunks into Supabase (its own
prerequisite is `sql/create_sql_tables.sql`, including the
`match_rag11_child_chunks` RPC).

In [1]:
# Cell #02
import os
import re
import time
import random

from dotenv import load_dotenv
from supabase import create_client, Client
import voyageai
import anthropic

load_dotenv()


def require_env(name: str) -> str:
    """Fetch an env var and fail with a clear, actionable message (naming
    the exact .env line to fill in) instead of a cryptic downstream error."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(
            f"{name} is empty in your .env file. Open .env in the RAG11 folder "
            f"and paste your actual value in after '{name}='."
        )
    return value


SUPABASE_URL = require_env("PUBLIC_SUPABASE_URL")
# This notebook only reads (RPC calls), it never writes, so the anon key is
# enough -- it can't be blocked by RLS the way an insert/upsert could be.
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "").strip() or require_env("PUBLIC_SUPABASE_ANON_KEY")
VOYAGE_API_KEY = require_env("VOYAGE_API_KEY")
ANTHROPIC_API_KEY = require_env("ANTHROPIC_API_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

EMBEDDING_MODEL = "voyage-3"           # must match stage1_2_eda_load_chunks.ipynb's EMBEDDING_MODEL
GENERATION_MODEL = "claude-sonnet-5"   # change here if your account uses a different Claude model id

print("Clients ready. Supabase project:", SUPABASE_URL)

Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co


**Cell #03**

## Retrieval

In [2]:
# Cell #04
def _retry(fn, *args, max_attempts: int = 5, base_delay: float = 2.0, **kwargs):
    """Exponential-backoff retry wrapper for flaky network calls (same
    pattern as stage1_2_eda_load_chunks.ipynb's _retry)."""
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_exc = e
            if attempt == max_attempts:
                break
            sleep_s = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"[retry] {getattr(fn, '__name__', fn)} attempt {attempt} "
                  f"failed ({e}); retrying in {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_exc


MIN_CONTEXT_CHUNKS = 3   # requirement: the LLM always sees at least this many retrieved chunks
NUM_CONTEXT_CHUNKS = 5   # how many chunks match_rag11_child_chunks actually returns per question

_PAGE_RANGE_RE = re.compile(r"Pages (\d+)-(\d+)")


def embed_query(text: str) -> list[float]:
    """Embed a question with input_type='query' -- Voyage's asymmetric
    embeddings expect the query side and the document side (used when the
    child chunks were embedded) to be encoded differently for best
    retrieval quality."""
    resp = _retry(voyage_client.embed, texts=[text], model=EMBEDDING_MODEL, input_type="query")
    return resp.embeddings[0]


def retrieve_chunks(question: str, match_count: int = NUM_CONTEXT_CHUNKS) -> list[dict]:
    """Return the `match_count` best-matching child rows for `question`,
    ordered by cosine distance ascending (closest first), via the
    match_rag11_child_chunks RPC from sql/create_sql_tables.sql."""
    query_embedding = embed_query(question)
    resp = _retry(lambda: supabase.rpc("match_rag11_child_chunks", {
        "query_embedding": query_embedding,
        "match_count": match_count,
    }).execute())
    return resp.data


def page_numbers_for_chunk(row: dict) -> list[int]:
    """Parse the inclusive page range out of a child row's contextual
    header, e.g. '[Source: foo.pdf | Section: Bar | Pages 12-14]' ->
    [12, 13, 14]. Every child chunk carries this header (see
    contextual_header() / write_chunks() in stage1_1_extract_and_chunk.ipynb),
    so this needs no extra DB lookup against the parent row."""
    text = row.get("rowJSON", {}).get("text", "")
    match = _PAGE_RANGE_RE.search(text)
    if not match:
        return []
    start_page, end_page = int(match.group(1)), int(match.group(2))
    return list(range(start_page, end_page + 1))

**Cell #05**

## Generation

In [3]:
# Cell #06
MAX_ANSWER_TOKENS = 800   # upper bound on how many tokens Claude's generated answer may use

SYSTEM_PROMPT = """You are a nutrition Q&A assistant. Answer strictly using \
the numbered excerpts provided in the user message -- do not rely on \
outside knowledge, and say plainly if the excerpts don't contain enough \
information to answer.

First decide whether the question has a clean Yes/No answer:
  - If it does, begin your reply with exactly this one line:
        Short answer: Yes
    or
        Short answer: No
    then a blank line, then the full explanation.
  - If the question has no clean Yes/No answer (it asks for a list, a \
description, a comparison -- a "what"/"how" question rather than an \
"is"/"does"/"can" one), skip the "Short answer" line entirely and just \
give the full explanation.

Keep the full explanation grounded in the excerpts -- refer to what they \
actually say rather than general nutrition knowledge."""


def build_context_block(chunks: list[dict]) -> str:
    parts = []
    for i, row in enumerate(chunks, start=1):
        source_key = row["rowJSON"].get("source_key", row["rowOwnerGUID"])
        parts.append(f"[Excerpt {i} -- {source_key}]\n{row['rowJSON']['text']}")
    return "\n\n".join(parts)


_SHORT_ANSWER_RE = re.compile(r"^\s*Short answer:\s*(Yes|No)\s*$", re.IGNORECASE | re.MULTILINE)


def extract_short_answer(answer_text: str) -> str | None:
    match = _SHORT_ANSWER_RE.search(answer_text)
    return match.group(1).capitalize() if match else None


# A compact, deliberately unsurprising English stopword list -- just enough
# to strip connective/filler words so grounding_words() below surfaces the
# actual nutrition terminology the answer and its source excerpts share.
_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "else", "for", "nor", "so",
    "as", "at", "by", "in", "into", "of", "on", "onto", "to", "with", "within",
    "from", "about", "above", "after", "again", "against", "all", "am", "any",
    "are", "because", "been", "before", "being", "below", "between", "both",
    "can", "cannot", "did", "do", "does", "doesn't", "doing", "down", "during",
    "each", "few", "further", "had", "has", "have", "having", "he", "her",
    "here", "hers", "herself", "him", "himself", "his", "how", "however", "i",
    "is", "it", "its", "itself", "just", "me", "more", "most", "my", "myself",
    "no", "not", "now", "off", "once", "only", "other", "our", "ours",
    "ourselves", "out", "over", "own", "same", "she", "should", "some", "such",
    "than", "that", "their", "theirs", "them", "themselves", "there", "these",
    "therefore", "they", "this", "those", "through", "thus", "too", "under",
    "until", "up", "very", "was", "we", "were", "what", "when", "where",
    "which", "while", "who", "whom", "why", "will", "would", "you", "your",
    "yours", "yourself", "yourselves",
}
_WORD_RE = re.compile(r"[A-Za-z']+")


def grounding_words(answer_text: str, chunks: list[dict], top_n: int = 12) -> list[str]:
    """Words the printed answer actually shares with its retrieved source
    excerpts -- a literal, checkable answer to "what words is this answer
    established on", rather than just trusting the system prompt's
    "answer only from the excerpts" instruction. Stopwords and 1-2 letter
    tokens are dropped; the rest are returned lowercased, deduplicated, in
    the order they first appear in the answer, capped at `top_n`."""
    excerpt_text = " ".join(row["rowJSON"]["text"] for row in chunks)
    excerpt_words = {w.lower() for w in _WORD_RE.findall(excerpt_text) if len(w) > 2}

    seen = set()
    shared = []
    for word in _WORD_RE.findall(answer_text):
        lower = word.lower()
        if lower in _STOPWORDS or len(lower) <= 2 or lower in seen:
            continue
        if lower in excerpt_words:
            seen.add(lower)
            shared.append(lower)
        if len(shared) >= top_n:
            break
    return shared


def ask_question(question: str, match_count: int = NUM_CONTEXT_CHUNKS) -> dict:
    """Retrieve chunks for `question`, ask Claude to answer from them only,
    and return a structured result -- including the short Yes/No call (if
    any), the array of source page numbers behind the answer, and the
    words the answer shares with those source excerpts."""
    chunks = retrieve_chunks(question, match_count=match_count)
    if len(chunks) < MIN_CONTEXT_CHUNKS:
        print(f"  [warn] only {len(chunks)} chunk(s) retrieved for {question!r} "
              f"(< {MIN_CONTEXT_CHUNKS}) -- answer may be under-supported.")

    context_block = build_context_block(chunks)
    user_message = f"{context_block}\n\nQuestion: {question}"

    resp = _retry(lambda: anthropic_client.messages.create(
        model=GENERATION_MODEL,
        max_tokens=MAX_ANSWER_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    ))
    answer_text = "".join(block.text for block in resp.content if block.type == "text")

    source_pages = sorted({page for row in chunks for page in page_numbers_for_chunk(row)})
    source_keys = sorted({row["rowJSON"].get("source_key", row["rowOwnerGUID"]) for row in chunks})

    return {
        "question": question,
        "short_answer": extract_short_answer(answer_text),   # "Yes" / "No" / None
        "answer": answer_text,
        "chunks_used": len(chunks),
        "source_pages": source_pages,   # array of page numbers backing this answer
        "source_keys": source_keys,
        "grounding_words": grounding_words(answer_text, chunks),  # words shared with the source excerpts
    }

**Cell #07**

## The 5 questions

A mix on purpose: some have a clean Yes/No answer (to exercise the
`Short answer:` formatting), some are open-ended (to check that format is
correctly skipped), and two are phrased as common oversimplifications so a
good answer has to lean on the nuance actually present in the source text
rather than a flat "yes"/"no".

In [4]:
# Cell #08
SAMPLE_QUESTIONS = [
    # Yes/No -- a clean textbook fact, good for checking the Short answer format.
    "Is vitamin C a water-soluble vitamin?",
    # Yes/No, but the honest answer is nuanced -- checks the model still
    # commits to Yes/No first and puts the nuance in the full explanation.
    "Does eating excess dietary protein get stored directly as body fat?",
    # Open-ended, multipart -- likely needs chunks from more than one source.
    "What roles do carbohydrates, proteins, and fats each play in providing energy to the body?",
    # Yes/No, commonly oversimplified -- tests whether retrieval surfaces the
    # nuance rather than a flat "yes, all saturated fat is bad".
    "Is saturated fat the only type of dietary fat linked to raised LDL cholesterol?",
    # Open-ended -- no yes/no framing at all.
    "What functions does dietary fiber serve in the digestive system?",
]

**Cell #09**

## Run all 5 questions

In [5]:
# Cell #10
results = []
for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    result = ask_question(question)
    results.append(result)

    print(f"  chunks used: {result['chunks_used']} (source(s): {', '.join(result['source_keys'])})")
    print(f"  source pages: {result['source_pages']}")
    if result["short_answer"]:
        print(f"  Short answer: {result['short_answer']}")
    print()
    print(result["answer"])
    print()
    grounding = ", ".join(result["grounding_words"]) or "(no shared terms found with the retrieved excerpts)"
    print(f"  established on: {grounding}")
    print("\n" + "-" * 80 + "\n")

Q: Is vitamin C a water-soluble vitamin?
  chunks used: 5 (source(s): source13, source17, source2, source4)
  source pages: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124]
  Short answer: Yes

Short answer: Yes

Excerpt 1 explicitly states that "Ascorbic acid is a water-soluble vitamin commonly known as vitamin C." This is further supported by Excerpt 2, which lists "Vitamin C (ascorbic acid)" under the category of water-soluble vitamins alongside the B-complex vitamins, and Excerpt 3, which states there are 10 water-soluble vitamins and includes "Vitamin C, also known as ascorbic acid" among them.

  established on: states, ascorbic, aci

**Cell #11**

## Summary table

In [6]:
# Cell #12
print(f"{'#':<3} {'short answer':<13} {'chunks':>7} {'pages':>6} {'sources':<12} question")
for i, r in enumerate(results, start=1):
    short = r["short_answer"] or "n/a"
    print(f"{i:<3} {short:<13} {r['chunks_used']:>7} {len(r['source_pages']):>6} "
          f"{','.join(r['source_keys']):<12} {r['question']}")

#   short answer   chunks  pages sources      question
1   Yes                 5     94 source13,source17,source2,source4 Is vitamin C a water-soluble vitamin?
2   No                  5     79 source11,source17,source3 Does eating excess dietary protein get stored directly as body fat?
3   n/a                 5     35 source17,source2,source3 What roles do carbohydrates, proteins, and fats each play in providing energy to the body?
4   No                  5     62 source17,source2 Is saturated fat the only type of dietary fat linked to raised LDL cholesterol?
5   n/a                 5     37 source17,source4,source8 What functions does dietary fiber serve in the digestive system?


**Cell #13**

## Save workspace to GitHub

Synchronize this notebook, answers, and any code changes to GitHub using  (with auto lock recovery and conflict resolution).

In [ ]:
# Cell #14
import os
import subprocess
import sys


def save_to_github(commit_msg: str = "Update stage2_ask_samples1.ipynb") -> bool:
    """Bulletproof save, commit, and push to GitHub (cleans stale locks & syncs automatically)."""
    env = dict(os.environ, NON_INTERACTIVE="1")
    res = subprocess.run(
        ["/bin/zsh", "save_to_github.command", commit_msg],
        capture_output=True,
        text=True,
        env=env,
    )
    print(res.stdout)
    if res.stderr:
        print(res.stderr, file=sys.stderr)
    return res.returncode == 0


save_to_github("stage2_ask_samples1.ipynb - answers verified and synced")
